#Initialization

In [0]:
%python
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col


#Read bronze table erp_cust_az12

In [0]:
%python
df = spark.table("workspace.bronze.erp_cust_az12")

#Silver Transformations

##Trimming

In [0]:
%python
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))


##Customer ID Cleanup

In [0]:
%python
df = (
    df.withColumn(
        "cid",
        F.when(col("cid").startswith("NAS"),
               F.substring(col("cid"), 4, F.length(col("cid"))))
               .otherwise(col("cid"))
    )
    )

##Gender Normalization

In [0]:
%python
df = (
    df.withColumn("gen",
                  F.when(F.upper(col("gen")).isin("FEMALE", "F"), "Female")
                  .when(F.upper(col("gen")).isin("MALE", "M"), "Male")
                  .otherwise("n/a")
                  )
)

##Birthdate validation

In [0]:
%python
df = (
    df.withColumn(
        "bdate", 
        F.when(col("bdate") > F.current_date(), None)
        .otherwise(col("bdate")))
)

In [0]:
%python
df.display()

##Renaming columns

In [0]:
%python
RENAME_map = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}

for old_name, new_name in RENAME_map.items():
    df = df.withColumnRenamed(old_name, new_name)

##Sanity checks of dataframe

In [0]:
%python
df.limit(10).display()

#Writing silver table

In [0]:
%python
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customers")

##Sanity checks of silver table

In [0]:
SELECT * FROM silver.erp_customers LIMIT 10